# Day 09 — OpenCV Temelleri ve Görüntü Ön İşleme
## OpenCV ile Piksel Erişimi, Filtreleme, Gri Tonlama ve Histogram Eşitleme

> **Aşama:** Faz 2 — Bilgisayarlı Görü (Day 09–15)
> **Resmi Staj Defteri Konusu:** OpenCV Temelleri ve Görüntü Ön İşleme (Yaprak 17 & 18)

### 1. Problem
Dokuma tezgâhı üzerindeki optik denetim kameralarından gelen ham görüntüler; ortam aydınlatma değişkenlikleri, optik bozulmalar ve tekstil tozu kaynaklı yüksek frekanslı gürültü içerir. Görüntüler filtrelenmeden ve kontrastı normalize edilmeden desen analizi yapılamaz.

### 2. Why the Problem Matters
OpenCV kütüphanesi C++ tabanlı optimize görüntü işleme çekirdeğiyle endüstride gerçek zamanlı görüntü işleme sağlar. CLAHE (Adaptive Histogram Equalization) ve Gaussian yumuşatma ile dokuma kusurlarının belirginliği artırılır.

### 3. Engineering Concepts
- **BGR vs RGB**: OpenCV'nin varsayılan BGR renk kanalı dizilimi.
- **Gaussian Blur**: Kenarları korurken sensor gürültüsünü filtreleme.
- **CLAHE (Contrast Limited Adaptive Histogram Equalization)**: Yerel piksellerde kontrastı artırma.

In [ ]:
# 4. Library / API Investigation
import cv2
import numpy as np
print(f"OpenCV Version: {cv2.__version__}")

In [ ]:
# 5. Minimal Implementation
import cv2
import numpy as np

class ImagePreprocessor:
    def __init__(self, target_size=(256, 256)):
        self.target_size = target_size

    def resize(self, img: np.ndarray) -> np.ndarray:
        return cv2.resize(img, self.target_size, interpolation=cv2.INTER_AREA)

    def to_grayscale(self, img: np.ndarray) -> np.ndarray:
        if len(img.shape) == 2:
            return img
        return cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)

    def enhance_contrast_clahe(self, gray_img: np.ndarray) -> np.ndarray:
        clahe = cv2.createCLAHE(clipLimit=2.0, tileGridSize=(8, 8))
        return clahe.apply(gray_img)

# Sentetik dokuma deseni üretimi
synthetic_carpet = np.zeros((300, 300, 3), dtype=np.uint8)
synthetic_carpet[:, :] = (120, 80, 50)
synthetic_carpet[50:250, 50:250] = (200, 150, 100)

preprocessor = ImagePreprocessor(target_size=(256, 256))
resized = preprocessor.resize(synthetic_carpet)
gray = preprocessor.to_grayscale(resized)
enhanced = preprocessor.enhance_contrast_clahe(gray)

print(f"Orijinal Boyut: {synthetic_carpet.shape} -> Yeniden Boyutlandırılmış: {resized.shape}")
print(f"Gri Tonlama Boyut: {gray.shape} | CLAHE Kontrast Güçlendirildi.")


In [ ]:
# 6. Experiment: CLAHE ile Kontrast Farkı
gray = preprocessor.to_grayscale(resized)
std_before = float(np.std(gray))
std_after = float(np.std(enhanced))
print(f"Ön İşleme Öncesi Standart Sapma: {std_before:.2f} | CLAHE Sonrası: {std_after:.2f}")

In [ ]:
# 7. Visualization
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(7, 3.5))
axes[0].imshow(cv2.cvtColor(resized, cv2.COLOR_BGR2RGB))
axes[0].set_title("Orijinal Sentetik Görsel")
axes[0].axis("off")

axes[1].imshow(enhanced, cmap="gray")
axes[1].set_title("Ön İşlenmiş & CLAHE Çıktısı")
axes[1].axis("off")
plt.tight_layout()
plt.show()

In [ ]:
# 8. Validation
assert enhanced.shape == (256, 256)
assert std_after >= std_before
print("Kontrast artışı ve ön işleme hattı başarıyla doğrulandı.")

In [ ]:
# 9. Failure Cases: Tek kanallı görselin tekrar griye dönüştürülmesi
already_gray = np.zeros((100, 100), dtype=np.uint8)
safe_gray = preprocessor.to_grayscale(already_gray)
assert safe_gray.shape == (100, 100)
print("Zaten gri olan görselde gereksiz dönüşüm başarıyla engellendi.")

### 10. Conclusions
OpenCV temelleri ve ön işleme zinciri (boyutlandırma, gri tonlama, gürültü giderme ve CLAHE) kurulmuş, sonraki günlerdeki renk uzayı ve segmentasyon adımları için optimize girdi hazırlanmıştır.